In [ ]:
import pandas as pd
from matplotlib.pyplot import show
from syslogging.logger import get_logger
from systems.jani.save_jani_system import futures_do_system


## How to use
- run `python systems/jani/save_jani_system.py <dotted path to config file>`
- for example:
- run `python systems/jani/save_jani_system.py systems.jani.dynamic_system_jani_v1_quick.yaml`
- that will run a system with the specified config, and save most of to a pickle file
- the name of the pickle file will be printed out on the last line
- copy that pickle file name into the cell below, as `SAVED_SYSTEM`
- then run the remaining cells
- NOTES:
  - each time you run the script above, it creates a new pickle file
  - pickle files are large, you may want to delete them from time to time
  - the pre-run doesn't include optimisation, so the notebook bit is still a little slow
  - in my experience the browser does stop responding for a bit, but does come back
  - you may want to beef up your VM if you want to make it faster
  - or learn how to do your development and backtesting on a faster machine

In [ ]:
CONFIG = "systems.jani.test_10instr_TrendCarry.yaml"
SAVED_SYSTEM = "systems.jani.test_10instr_TrendCarry-2024-12-01_123241.pck"

#CONFIG = "systems.jani.dynamic_system_jani_v1_quick.yaml"
#SAVED_SYSTEM = "systems.jani.dynamic_system_jani_v1_quick-2024-11-29_162505.pck"

#CONFIG = "systems.jani.dynamic_system_jani_v1_backtest_10instr.yaml"
#SAVED_SYSTEM = "systems.jani.dynamic_system_jani_v1_backtest_10instr-2024-11-30_220353.pck"

#CONFIG = "systems.jani.dynamic_system_jani_v1_backtest_RobsJumbo.yaml"
#SAVED_SYSTEM = "systems.jani.dynamic_system_jani_v1_backtest_RobsJumbo-2024-12-01_160329.pck"

log = get_logger("backtest")


In [ ]:
%set_env PYSYS_PRIVATE_CONFIG_DIR=/home/alpha/pst_private/prod
#%set_env PYSYS_PRIVATE_CONFIG_DIR=/Users/ageach/Dev/work/j1404/pst_private/dev_andy

log.info(f"Loading system from {SAVED_SYSTEM}")
system = futures_do_system()
system.cache.get_items_with_data()
system.cache.unpickle(SAVED_SYSTEM)
system.cache.get_items_with_data()

portfolio = system.accounts.optimised_portfolio()
portfolio_percent = system.accounts.portfolio().percent


## performance

In [ ]:
system.config.use_SR_costs = False
perf_unrounded = system.accounts.portfolio(roundpositions=False).percent
perf_rounded = system.accounts.portfolio(roundpositions=True).percent
perf_optimised = system.accounts.optimised_portfolio().percent

performance = pd.concat([perf_unrounded.curve(), perf_rounded.curve(), perf_optimised.curve()], axis=1)
performance.columns = ["unrounded", "rounded", "optimised"]

print(f"Stats as %: {portfolio_percent.stats()}")
performance.plot(figsize=(15,9), title="Performance")
show()


## summary stats

In [ ]:
corr = pd.concat([perf_unrounded.curve(), perf_optimised.curve()], axis=1)
sharpe_gross = system.accounts.optimised_portfolio().gross.sharpe()
sharpe_net = system.accounts.optimised_portfolio().net.sharpe()
sr_cost_loss = sharpe_gross - sharpe_net
turnover = system.accounts.total_portfolio_level_turnover()

print(f"Unrounded v optimised portfolio returns correlation: {round(corr.corr().iloc[0, 1], 5)}")
print(f"Sharpe gross: {round(sharpe_gross, 3)}")
print(f"Sharpe net: {round(sharpe_net, 3)}")
print(f"Sharpe gross net difference: {round(sr_cost_loss, 3)} (or, in basis points: ~{round(sr_cost_loss * 100)})")
print(f"Portfolio level turnover: {round(turnover, 2)}")


## costs

In [ ]:
unrounded = system.accounts.portfolio(roundpositions=False)
rounded = system.accounts.portfolio()
optimised = system.accounts.optimised_portfolio()

costs = pd.concat([unrounded.costs.curve(), rounded.costs.curve(), optimised.costs.curve()], axis=1)
costs.columns = ["unrounded", "rounded", "optimised"]

costs.plot(figsize=(15,9), title="Costs")
show()


## costs v performance
- as per https://qoppac.blogspot.com/2021/11/mr-greedy-and-tale-of-minimum-tracking.html

In [ ]:
optimised = system.accounts.optimised_portfolio().percent.net
costs = optimised.costs.curve()
costs = costs * -10
costs_v_perf = pd.concat([optimised.curve(), costs], axis=1)
costs_v_perf.columns = ["Net performance %", "Costs (x -1.0)"]
costs_v_perf.plot(figsize=(15,9))
show()


## Risk

In [ ]:
unrounded = system.risk.get_portfolio_risk_for_original_positions()
rounded = system.risk.get_portfolio_risk_for_original_positions_rounded_buffered()
optimised = system.risk.get_portfolio_risk_for_optimised_positions()
risk = pd.concat([unrounded, rounded, optimised], axis=1)
risk.columns = ["unrounded", "rounded", "optimised"]
risk.tail(1500).plot(figsize=(15,9), title="Risk")
show()


## positions plot

In [ ]:
for instr in system.portfolio.get_instrument_list():
#for instr in system.data.get_instrument_list():
    #not_pos = system.portfolio.get_notional_position(instr)
    unrounded = system.portfolio.accounts_stage.get_buffered_position(
        instr, roundpositions=False
    )
    rounded = system.portfolio.accounts_stage.get_buffered_position(
        instr, roundpositions=True
    )
    optimised = system.accounts.get_optimised_position_df()[instr]
    pos = pd.concat([unrounded, rounded, optimised], axis=1)
    pos.columns = ["unrounded", "rounded", "optimised"]
    pos.plot(figsize=(15,9), title=f"{instr}")
    #pos["2000-01-01":].plot(title=f"{instr} (min bet {min_bet})", figsize=(15,9))
    #pos.tail(500).plot(figsize=(15,9), title=f"{instr}")
    show()



## instrument count by date

In [ ]:
import datetime
price_dict = dict(
    [
        (
            instrument_code,
            system.data.get_raw_price_from_start_date(
                instrument_code, datetime.datetime(1940, 1, 1)
            ),
        )
        for instrument_code in system.portfolio.get_instrument_list()
    ]
)

prices = pd.DataFrame(price_dict)
prices.ffill(inplace=True)
prices["instr_count"] = prices.notna().sum(axis=1)
prices["instr_count"]["1959-01-01":"2025-01-01"].plot(title="Instrument count (raw)", figsize=(15,9))
show()


## How many different instruments do we hold at once?

In [ ]:
positions = system.accounts.get_optimised_position_df()
positions["pos_count"] = positions.ne(0).sum(axis=1)
positions["pos_count"].plot(title="Position count", figsize=(15,9))
show()
